# 2 Urban surface parameters
## 2.1 Preprocessing: Update Building Polygons with Maximum Height
**Join the Height Table**
1. Join the processed CSV file containing the maximum building heights to the original building polygon shapefile using the appropriate unique identifier.
2. Export the joined layer as a new shapefile or feature class.
**Remove Lower-Height Polygons**
3. Delete polygons that do not contain any joined height values, as these correspond to duplicate building footprints with lower heights.
The result is `bldg_height_highest_filtered.shp`. Use this to generate roof area (2.3)


## 2.2 Polygon to line
**Generate Building Boundaries (sides)**
1. Densify (arcgis or QGIS). Generate vertices every 1m
2. Fix geometry (QGIS)
3. Snap geometry to layer (Behavior: 3. Prefer aligning nodes, don't insert new vertices). Tolerance: 1 m
4. Repair geometry (arcgis)
5. **Polygon to Raster** 0.2m resolution (20 cm)
6. **Raster calclulator** Height valuex * 100 (Heights are now in cm). This step is important to transform value into integer then Raster to polygon
7. **Integer** 
8. **Raster to Polygon**, Simplify polygons (Reduce the length of the perimeter, close to the original)
9. **Polygon to Line** Uncheck "Identify and store polygon neighboring information". Name `PTL_attributes`
10. Rerun **Polygon to Line**  and check "Identify and store polygon neighboring information" --> `LEFT_FID` and `RIGHT_FID` `PTL_neighbors`
11. **Intersect** between `PTL_attributes` and `FTL_neighbors` to keep original attributes and neighbor attributes
12. Create ne column named "line_ID" = FID + 1

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from dbfread import DBF


In [2]:
# Convert the line-feature DBF table to CSV.
dbf_path = Path(r"C:\Users\percy\Documents\studies\visual_studio_code\big_files\bldg_topology\polygon\bldg_int_rtp_line_attribures_neighbors_intersect.dbf")
out_csv = dbf_path.with_suffix(".csv")

if not dbf_path.exists():
    raise FileNotFoundError(f"DBF not found: {dbf_path}")

table = DBF(str(dbf_path), load=True, raw=False)
bldg_height_line = pd.DataFrame(iter(table))
bldg_height_line = bldg_height_line[["line_ID", "LEFT_FID", "RIGHT_FID", "gridcode"]]

# Keep the original order
bldg_height_line["original_order"] = bldg_height_line.index

# Sort by line_ID in ascending order
bldg_height_line = bldg_height_line.sort_values("line_ID").reset_index(drop=True)

# Create the new column
bldg_height_line["line_ID_a"] = 0

current_id = 1
previous_pair = None

for i in range(len(bldg_height_line)):

    left = bldg_height_line.loc[i, "LEFT_FID"]
    right = bldg_height_line.loc[i, "RIGHT_FID"]

    # Rule 1: LEFT_FID == -1
    if left == -1:
        bldg_height_line.loc[i, "line_ID_a"] = current_id
        current_id += 1

        # Reset the previous pair
        previous_pair = None

    # Rule 2: LEFT_FID != -1
    else:
        current_pair = (left, right)

        if current_pair == previous_pair:
            # Continue using the previous integer
            bldg_height_line.loc[i, "line_ID_a"] = current_id - 1
        else:
            # Assign a new integer
            bldg_height_line.loc[i, "line_ID_a"] = current_id
            current_id += 1

        previous_pair = current_pair

# Restore the original row order
bldg_height_line = bldg_height_line.sort_values("original_order").drop(columns="original_order")

# Save the result
bldg_height_line.to_csv("output.csv", index=False)

print(bldg_height_line.head())


: 

In [ ]:

# Create the new columns
bldg_height_line["left_height"] = 0.0
bldg_height_line["right_height"] = 0.0

# Case 1: LEFT_FID == -1
mask = bldg_height_line["LEFT_FID"] == -1
bldg_height_line.loc[mask, "left_height"] = 0
bldg_height_line.loc[mask, "right_height"] = bldg_height_line.loc[mask, "gridcode"]

# Case 2: LEFT_FID != -1
for group_id, group in bldg_height_line[~mask].groupby("line_ID_a"):

    # Make sure the rows are ordered by line_ID_a
    group = group.sort_values("line_ID_a")

    if len(group) >= 2:
        first_height = group.iloc[0]["gridcode"]
        second_height = group.iloc[1]["gridcode"]

        bldg_height_line.loc[group.index, "left_height"] = first_height
        bldg_height_line.loc[group.index, "right_height"] = second_height

    elif len(group) == 1:
        # Optional: if only one row exists in the group
        bldg_height_line.loc[group.index, "left_height"] = group.iloc[0]["gridcode"]
        bldg_height_line.loc[group.index, "right_height"] = 0

# Export the line-feature table with left_height to CSV for further analysis.
try:
    bldg_height_line.to_csv(out_csv, index=False)
    saved_csv = out_csv
except PermissionError:
    saved_csv = dbf_path.with_name(f"{dbf_path.stem}_with_left_height.csv")
    bldg_height_line.to_csv(saved_csv, index=False)    
 

In [ ]:
# Keep unique line_ID_a for the line-feature table based on the rules provided.
# Sort by line_ID_a
bldg_height_line = bldg_height_line.sort_values("line_ID_a")

# For duplicate line_ID_a values, keep only the first (top) row
bldg_height_line = bldg_height_line.drop_duplicates(subset="line_ID_a", keep="first")

# Reset the index (optional)
bldg_height_line = bldg_height_line.reset_index(drop=True)
   

In [ ]:
# Create a new column with the absolute difference
bldg_height_line["abs_height"] = (bldg_height_line["right_height"] - bldg_height_line["left_height"]).abs()

# Export the line-feature table with left_height to CSV for further analysis.
try:
    bldg_height_line.to_csv(out_csv, index=False)
    saved_csv = out_csv
except PermissionError:
    saved_csv = dbf_path.with_name(f"{dbf_path.stem}_with_left_height.csv")
    bldg_height_line.to_csv(saved_csv, index=False) 

### 2.1.2 Extract the Length of Each Line
1. Join the exported CSV file to `bldg_rtp_PTL_intersect.shp` using the `line_ID` and `line_ID`. 
2. Export the joined layer as a new shapefile named:`bldg_rtp_PTL_intersect_JOINED.shp` (LAST STEP)
3. Filter `line_ID_a` <> 0 to remove duplicates and export to `bldg_rtp_PTL_intersect_JOINED_filtered.shp`
4. Intersect with `lcz_250m_lc` and save as `bldg_line_JOINED_filtered_intersect_lcz.shp`
5. Create a new field named `length`.
6. Use **Calculate Geometry** to compute the length of each line segment and store the result in the `length` field.
